# Dataset Exploration
Statistical details and visualisations for all 3 brain MRI datasets before preprocessing.

## 1. Setup

In [ ]:
!pip install numpy pandas matplotlib seaborn pillow --quiet

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import defaultdict

try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

# ── Paths ─────────────────────────────────────────────────────────
if ON_COLAB:
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/brain_mri_raw')
else:
    DATA_DIR = Path('./data')

print(f'Data directory: {DATA_DIR}')
print(f'Running on: {"Google Colab" if ON_COLAB else "local environment"}')

## 2. Scan Datasets
Collect image paths, class labels, splits and resolutions for all 3 datasets.

In [ ]:
# ── Dataset configs ────────────────────────────────────────────────
# Dataset 3 has no train/test split — all images are in class folders directly
DATASETS = {
    'Dataset 1 — Brain Tumor Classification (MRI)': {
        'path':       DATA_DIR / 'dataset1',
        'has_splits': True,
        'splits':     ['Training', 'Testing'],
    },
    'Dataset 2 — Brain Tumor MRI Dataset': {
        'path':       DATA_DIR / 'dataset2',
        'has_splits': True,
        'splits':     ['Training', 'Testing'],
    },
    'Dataset 3 — Brain Tumor MRI Images 44 Classes': {
        'path':       DATA_DIR / 'dataset3',
        'has_splits': False,
        'splits':     [None],
    },
}


def scan_dataset(cfg):
    """Walk dataset folder and return a DataFrame with one row per image."""
    rows = []
    base = cfg['path']
    for split in cfg['splits']:
        root = base / split if split else base
        for cls_folder in sorted(root.iterdir()):
            if not cls_folder.is_dir(): continue
            for img_path in cls_folder.iterdir():
                if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
                try:
                    w, h = Image.open(img_path).size
                    rows.append({
                        'path':  str(img_path),
                        'class': cls_folder.name,
                        'split': split if split else 'all',
                        'width': w,
                        'height': h,
                        'pixels': w * h,
                    })
                except Exception:
                    pass
    return pd.DataFrame(rows)


dfs = {}
for name, cfg in DATASETS.items():
    print(f'Scanning {name} ...')
    dfs[name] = scan_dataset(cfg)
    print(f'  {len(dfs[name]):,} images found')

print('\nDone')

## 3. Summary Statistics

In [ ]:
for name, df in dfs.items():
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    print(f'  Total images   : {len(df):,}')
    print(f'  Classes        : {df["class"].nunique()}')
    print(f'  Splits         : {df["split"].unique().tolist()}')
    print(f'\n  Resolution:')
    print(f'    Width  — min: {df["width"].min()}  max: {df["width"].max()}  mean: {df["width"].mean():.0f}')
    print(f'    Height — min: {df["height"].min()}  max: {df["height"].max()}  mean: {df["height"].mean():.0f}')
    print(f'\n  Images per class:')
    counts = df.groupby('class').size().sort_values(ascending=False)
    for cls, cnt in counts.items():
        print(f'    {cls:<35} {cnt:>5}')

## 4. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Class Distribution per Dataset', fontweight='bold', fontsize=14)

for ax, (name, df) in zip(axes, dfs.items()):
    counts = df.groupby('class').size().sort_values(ascending=False)
    short_name = name.split('—')[0].strip()
    sns.barplot(x=counts.values, y=counts.index, ax=ax, palette='Blues_r')
    ax.set_title(short_name, fontweight='bold')
    ax.set_xlabel('Number of images')
    ax.set_ylabel('')
    # Annotate bars
    for i, v in enumerate(counts.values):
        ax.text(v + 5, i, str(v), va='center', fontsize=8)

plt.tight_layout()
plt.savefig(DATA_DIR / 'exploration_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Train / Test Split Distribution (Datasets 1 & 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution by Split', fontweight='bold', fontsize=14)

for ax, (name, df) in zip(axes, list(dfs.items())[:2]):
    pivot = df.groupby(['class', 'split']).size().unstack(fill_value=0)
    pivot.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
    ax.set_title(name.split('—')[0].strip(), fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Number of images')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Split')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'exploration_split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Resolution Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Resolution Distribution per Dataset', fontweight='bold', fontsize=14)

for col, (name, df) in enumerate(dfs.items()):
    short = name.split('—')[0].strip()

    # Width histogram
    axes[0, col].hist(df['width'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[0, col].axvline(224, color='red', linestyle='--', linewidth=1.5, label='224px (target)')
    axes[0, col].set_title(f'{short}\nWidth', fontweight='bold')
    axes[0, col].set_xlabel('Width (px)')
    axes[0, col].set_ylabel('Count')
    axes[0, col].legend(fontsize=8)
    axes[0, col].grid(alpha=0.3)

    # Height histogram
    axes[1, col].hist(df['height'], bins=30, color='coral', edgecolor='white', alpha=0.8)
    axes[1, col].axvline(224, color='red', linestyle='--', linewidth=1.5, label='224px (target)')
    axes[1, col].set_title(f'Height')
    axes[1, col].set_xlabel('Height (px)')
    axes[1, col].set_ylabel('Count')
    axes[1, col].legend(fontsize=8)
    axes[1, col].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / 'exploration_resolution_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Pixel Intensity Distribution
Samples 50 images per dataset to show the distribution of grayscale pixel values.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Pixel Intensity Distribution (50 sampled images per dataset)', fontweight='bold')

for ax, (name, df) in zip(axes, dfs.items()):
    all_pixels = []
    sample = df.sample(min(50, len(df)), random_state=42)
    for _, row in sample.iterrows():
        img = np.array(Image.open(row['path']).convert('L'))  # convert to grayscale
        all_pixels.append(img.flatten())
    all_pixels = np.concatenate(all_pixels)

    ax.hist(all_pixels, bins=64, color='slategray', edgecolor='none', alpha=0.8)
    ax.set_title(name.split('—')[0].strip(), fontweight='bold')
    ax.set_xlabel('Pixel value (0–255)')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.3)
    ax.text(0.97, 0.95, f'mean={all_pixels.mean():.1f}\nstd={all_pixels.std():.1f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig(DATA_DIR / 'exploration_intensity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Sample Images per Class

In [ ]:
# ── Datasets 1 & 2 (4 classes) ────────────────────────────────────
for name, df in list(dfs.items())[:2]:
    classes = df['class'].unique()
    n_cols  = 5   # samples per class
    fig, axes = plt.subplots(len(classes), n_cols, figsize=(n_cols * 2.5, len(classes) * 2.5))
    fig.suptitle(f'Sample Images — {name.split("—")[0].strip()}', fontweight='bold', fontsize=13)

    for row, cls in enumerate(sorted(classes)):
        samples = df[df['class'] == cls].sample(min(n_cols, len(df[df['class'] == cls])), random_state=42)
        for col, (_, img_row) in enumerate(samples.iterrows()):
            ax = axes[row, col]
            ax.imshow(Image.open(img_row['path']).convert('L'), cmap='gray')
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(cls, fontsize=9, rotation=0, labelpad=60, va='center')

    plt.tight_layout()
    safe = name.replace(' ', '_').replace('—', '-')[:30]
    plt.savefig(DATA_DIR / f'exploration_samples_{safe}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Dataset 3 (44 classes — show 1 sample per class) ──────────────
name = 'Dataset 3 — Brain Tumor MRI Images 44 Classes'
df   = dfs[name]
classes = sorted(df['class'].unique())
n_cols  = 8
n_rows  = -(-len(classes) // n_cols)   # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2.5))
fig.suptitle('Sample Images — Dataset 3 (1 per class)', fontweight='bold', fontsize=13)
axes_flat = axes.flatten()

for i, cls in enumerate(classes):
    sample = df[df['class'] == cls].sample(1, random_state=42).iloc[0]
    axes_flat[i].imshow(Image.open(sample['path']).convert('L'), cmap='gray')
    axes_flat[i].set_title(cls, fontsize=6)
    axes_flat[i].axis('off')

# Hide unused axes
for j in range(len(classes), len(axes_flat)):
    axes_flat[j].axis('off')

plt.tight_layout()
plt.savefig(DATA_DIR / 'exploration_samples_dataset3.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Class Imbalance Summary

In [ ]:
for name, df in dfs.items():
    counts  = df.groupby('class').size()
    ratio   = counts.max() / counts.min()
    std_pct = (counts.std() / counts.mean()) * 100
    print(f'{name.split("—")[0].strip()}')
    print(f'  Most common class  : {counts.idxmax()} ({counts.max():,} images)')
    print(f'  Least common class : {counts.idxmin()} ({counts.min():,} images)')
    print(f'  Imbalance ratio    : {ratio:.2f}x')
    print(f'  Coefficient of variation: {std_pct:.1f}%')
    if ratio > 2:
        print(f'  ⚠ Notable class imbalance — may affect training')
    else:
        print(f'  ✓ Classes are reasonably balanced')
    print()